In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
data = pd.read_csv('imdb_top_1000.csv')
print(data.head())
print(data.info())

                                         Poster_Link  \
0  https://m.media-amazon.com/images/M/MV5BMDFkYT...   
1  https://m.media-amazon.com/images/M/MV5BM2MyNj...   
2  https://m.media-amazon.com/images/M/MV5BMTMxNT...   
3  https://m.media-amazon.com/images/M/MV5BMWMwMG...   
4  https://m.media-amazon.com/images/M/MV5BMWU4N2...   

               Series_Title Released_Year Certificate  Runtime  \
0  The Shawshank Redemption          1994           A  142 min   
1             The Godfather          1972           A  175 min   
2           The Dark Knight          2008          UA  152 min   
3    The Godfather: Part II          1974           A  202 min   
4              12 Angry Men          1957           U   96 min   

                  Genre  IMDB_Rating  \
0                 Drama          9.3   
1          Crime, Drama          9.2   
2  Action, Crime, Drama          9.0   
3          Crime, Drama          9.0   
4          Crime, Drama          9.0   

                         

In [3]:
data= data.drop_duplicates()
data.shape

(1000, 16)

In [4]:
data=data.reset_index()

In [5]:
def get_features(data):
    features = []
    for i in range(data.shape[0]):
        feature = (data['Series_Title'].iloc[i] + ', ' + 
                   data['Genre'].iloc[i] + ', ' + 
                   data['Overview'].iloc[i] + ', ' + 
                   data['Director'].iloc[i] + ', ' + 
                   data['Star1'].iloc[i] + ', ' + 
                   data['Star2'].iloc[i] + ', ' + 
                   data['Star3'].iloc[i] + ', ' + 
                   data['Star4'].iloc[i])
        features.append(feature)
    return features


In [6]:
important_features = get_features(data)
important_features

['The Shawshank Redemption, Drama, Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency., Frank Darabont, Tim Robbins, Morgan Freeman, Bob Gunton, William Sadler',
 "The Godfather, Crime, Drama, An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son., Francis Ford Coppola, Marlon Brando, Al Pacino, James Caan, Diane Keaton",
 'The Dark Knight, Action, Crime, Drama, When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice., Christopher Nolan, Christian Bale, Heath Ledger, Aaron Eckhart, Michael Caine',
 'The Godfather: Part II, Crime, Drama, The early life and career of Vito Corleone in 1920s New York City is portrayed, while his son, Michael, expands and tightens his grip on the family crime syndicate., Francis Ford Coppola, Al Pacino, Ro

In [7]:
len(important_features)

1000

In [8]:
data['important_features'] = get_features(data)

In [9]:
data['important_features'].head()

0    The Shawshank Redemption, Drama, Two imprisone...
1    The Godfather, Crime, Drama, An organized crim...
2    The Dark Knight, Action, Crime, Drama, When th...
3    The Godfather: Part II, Crime, Drama, The earl...
4    12 Angry Men, Crime, Drama, A jury holdout att...
Name: important_features, dtype: object

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['important_features'])
tfidf_matrix.shape

(1000, 10231)

In [11]:
#generating the cosine similarity
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               1000 non-null   int64  
 1   Poster_Link         1000 non-null   object 
 2   Series_Title        1000 non-null   object 
 3   Released_Year       1000 non-null   object 
 4   Certificate         899 non-null    object 
 5   Runtime             1000 non-null   object 
 6   Genre               1000 non-null   object 
 7   IMDB_Rating         1000 non-null   float64
 8   Overview            1000 non-null   object 
 9   Meta_score          843 non-null    float64
 10  Director            1000 non-null   object 
 11  Star1               1000 non-null   object 
 12  Star2               1000 non-null   object 
 13  Star3               1000 non-null   object 
 14  Star4               1000 non-null   object 
 15  No_of_Votes         1000 non-null   int64  
 16  Gross  

In [13]:
indices = pd.Series(data.index, index = data['Series_Title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    #Get the pairwise similarity scores of all movies with that movie
    sim_scores= list(enumerate(cosine_sim[idx]))
    #Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda X: X[1], reverse=True)
    sim_scores = sim_scores[1:6]
    movie_indices = [i[0] for i in sim_scores]
    #return the top 5 most similar movies
    movies = data['Series_Title'].iloc[movie_indices]
    id=data['index'].iloc[movie_indices]
    dict={"Movies":movies, "id": id}
    final_df=pd.DataFrame(dict)
    final_df.reset_index(drop=True,inplace=True)
    return final_df

In [14]:
get_recommendations("Fight Club")

,Movies,id
0,Se7en,27
1,The Curious Case of Benjamin Button,628
2,Snatch,96
3,Birdman or (The Unexpected Virtue of Ignorance),733
4,American History X,40


In [ ]:
import pickle
pickle.dump(new, op)